# 2. Generating and running kicked-Ising circuits

This notebook is analogous to the previous, except we randomly generate and run kicked-Ising circuits to test the prediction of our benchmarking circuits. 

In [1]:
# Initial imports:

from IPython.display import clear_output
import sys
sys.path.append('../')
import time
import numpy as np

# Our custom functions
print('Importing fccb...')
import fccb

clear_output(wait=True)
print('Importing Qiskit bits...')

# Qiskit functions and estimator primitives
from qiskit import QuantumCircuit, transpile
from qiskit.transpiler.passes import RemoveBarriers
from qiskit.primitives import Estimator as Estimator_sim
from qiskit.quantum_info import Pauli
from qiskit.circuit import Parameter
from qiskit.circuit.library import HamiltonianGate
from qiskit_aer.primitives import Estimator as Estimator_Aer
estimator_clifford = Estimator_Aer(backend_options=dict(method="stabilizer"))
estimator_exact = Estimator_sim()

# Qiskit Runtime
clear_output(wait=True)
print('Initialising QiskitRuntimeService...')
from qiskit_ibm_runtime import QiskitRuntimeService, EstimatorV2, Batch
service = QiskitRuntimeService(name='MY-ACCOUNT-HERE')

# Setup a backend and import its properties
clear_output(wait=True)
print('Importing backend...')
hardware = service.get_backend('ibm_brisbane')
basis_gates = hardware.basis_gates
coupling_map = hardware.coupling_map

# Three circuit layers required to implement 2-qubit interactions
Ls = np.load('../layers.npy')

clear_output(wait=True)
print('Done!')

Done!


Functions for transpiling KI circuits for classical simulation and for running on hardware:

In [2]:
def IBM_experiment_circ(device_qubits, n_layers):
    
    N = len(device_qubits)
    qc = QuantumCircuit(N)
    theta = Parameter('theta')
    
    for layer in range(n_layers):
        
        # Rx on each qubit
        for i in range(N):
            qc.rx(theta, i)
            
            
        # Rzz over 3 layers
        for Li in Ls:
            
            for q0, q1 in Li:
                if q0 in device_qubits and q1 in device_qubits:
                    i0, i1 = device_qubits.index(q0), device_qubits.index(q1)
                    qc.rzz(-np.pi/2, i0, i1)

    return qc, theta

In [3]:
rzz_trans = QuantumCircuit(2)

P = Pauli('ZZ')
U = HamiltonianGate(P,time=-np.pi/4)
rzz_trans.append(U,[0,1])
rzz_trans = transpile(rzz_trans, basis_gates=basis_gates)

def transpile_IBM_circ(device_qubits, n_layers, theta):
    
    rx_trans = QuantumCircuit(1)
    rx_trans.rx(theta,0)
    rx_trans = transpile(rx_trans, basis_gates=['ecr','id','rz','sx','x'])

    qc = QuantumCircuit(127)
    
    for l in range(n_layers):
    
        # Rx on each qubit
        for i in device_qubits:
            qc.compose(rx_trans, [i], inplace=True)

        qc.barrier()
        
        # Rzz over 3 layers
        for Li in Ls:
            for q0, q1 in Li:
                if q0 in device_qubits and q1 in device_qubits:             
                    qc.compose(rzz_trans, [q0, q1], inplace=True)

        qc.barrier()
        
    return qc

In [4]:
# Check we have a single ECR gate per RZZ gate
rzz_trans.draw()

┌──────────┐                  ┌──────┐   ┌───┐   ┌──────────┐»
q_0: ─┤ Rz(-π/2) ├──────────────────┤0     ├───┤ X ├───┤ Rz(-π/2) ├»
     ┌┴──────────┴┐┌────┐┌─────────┐│  Ecr │┌──┴───┴──┐└──┬────┬──┘»
q_1: ┤ Rz(1.8678) ├┤ √X ├┤ Rz(π/2) ├┤1     ├┤ Rz(π/2) ├───┤ √X ├───»
     └────────────┘└────┘└─────────┘└──────┘└─────────┘   └────┘   »
«                    
«q_0: ───────────────
«     ┌─────────────┐
«q_1: ┤ Rz(-1.8678) ├
«     └─────────────┘

Generate 900 sets of parameters (random number of qubits, random connected set of qubits to run circuit on and random number of circuit layers). Generate the corresponding circuit, simulate and measure $\langle Z_{62} \rangle$ classically and record the result. 

In [5]:
def generate_experiment(coupling_map, N, n_layers):

    # Pick device qubits
    central_qubit = 62
    # N = np.random.choice([i for i in range(1,10)])
    device_qubits = list([central_qubit])
    for i in range(N-1):
        connected_qubits_list = []
        for q in device_qubits:
            connected_qubits_list = connected_qubits_list + fccb.connected_qubits(q, coupling_map)
        connected_qubits_list = [q for q in connected_qubits_list if q not in device_qubits]
        new_qubit = np.random.choice(connected_qubits_list)
        device_qubits.append(new_qubit)

    device_qubits.sort()

    # Generate Pauli
    i = device_qubits.index(central_qubit)
    pauli_to_measure = 'I'*i + 'Z' + 'I'*(N-i-1)

    return device_qubits, pauli_to_measure

In [6]:
n_reps = 900
experiment_params = [None]*n_reps
exact_values = []

In [7]:
# # Generate experimental parameters

# for rep in range(n_reps):

#     n_qubits = np.random.choice([i+1 for i in range(16)])
#     n_layers = np.random.choice([i+1 for i in range(15)])
#     angle = np.random.uniform(0,np.pi/4)
    
#     if experiment_params[rep] is None:


#         device_qubits, pauli_to_measure = generate_experiment(coupling_map, n_qubits, n_layers)

#         qc, theta = IBM_experiment_circ(device_qubits, n_layers)
#         qc_theta = qc.assign_parameters({theta: angle})
#         exact_value = estimator_exact.run(qc_theta, pauli_to_measure[::-1]).result().values[0]
#         exact_values.append(result)

#         experiment_params[rep] = [device_qubits, pauli_to_measure, n_layers, angle, exact_value]

#     clear_output(wait=True)
#     print('Done for repetition ', rep+1)
#     time.sleep(0.01)

In [8]:
# experiment_params = np.array(experiment_params, dtype='object')
# np.save('KI_parameters.npy', experiment_params)

In [9]:
experiment_params = np.load('KI_parameters.npy', allow_pickle=True)
exact_values = [x[4] for x in experiment_params]

Check that there are very few expectation values close to zero: 

In [10]:
np.mean([abs(x) for x in exact_values])

0.8402189229284815

In [11]:
np.sum([1 for x in exact_values if abs(x) < 0.3])

21

Transpile circuits for submission to hardware:

In [12]:
# Generate list of expanded QuantumCircuits objects and expanded Pauli strings
# for submission to hardware

circuits_list = []
observables_list = []

rb = RemoveBarriers()

for rep in range(n_reps):

    device_qubits, pauli_to_measure, n_layers, angle = experiment_params[rep]
    qc = transpile_IBM_circ(device_qubits, n_layers, angle)
    
    circuits_list.append(qc)
    observable = fccb.expand_pauli(pauli_to_measure, device_qubits, 127)[::-1]
    observables_list.append(observable)

    clear_output(wait=True)
    print('Circuit', rep+1, 'transpiled')

Circuit 900 transpiled


Submit jobs:

In [13]:
# Max circuits per job is 300 on open plan. Hence split circuits into batches of up to 300
circuits_list_of_lists = [0]*3
circuits_list_of_lists[0] = circuits_list[0:300]
circuits_list_of_lists[1] = circuits_list[300:600]
circuits_list_of_lists[2] = circuits_list[600:900]

observables_list_of_lists = [0]*3
observables_list_of_lists[0] = observables_list[0:300]
observables_list_of_lists[1] = observables_list[300:600]
observables_list_of_lists[2] = observables_list[600:900]

In [14]:
# Submit data to IBM backend

# Can queue up to 3 at once:
job_ids = []

with Batch(service=service, backend=hardware) as batch:
    
        options = set_options(qem_level=0)
        estimator_noisy = Estimator(batch, options=options)
        
        for i in range(3):
            job = estimator_noisy.run(circuits=circuits_list_of_lists[i], observables=observables_list_of_lists[i])
            job_id = job.job_id()
            job_ids.append(job_id)
            print('Job ID ' + str(i+1) + ': ', job_id)

Import and save job results as tuples of (measured expectation value, measured variance):

In [15]:
job_ids_list = ['','','']

In [16]:
job_data = []
for job_id in job_ids_list:
    vals = service.job(job_id).result().values
    variances = [float(x['variance']) for x in service.job(job_id).result().metadata]
    for i in range(300):
        job_data.append((vals[i],variances[i]))
    print('job results imported')

job results imported
job results imported
job results imported


In [17]:
np.save('KI_job_data_brisbane.npy',job_data)

We process and plot this data in Notebook 3. 